### Decision api test

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")
AZURE_API_KEY = os.getenv("AZURE_API_KEY")
AZURE_DEPLOYMENT_NAME = os.getenv("AZURE_DEPLOYMENT_NAME")

#### BASIC SETUP

In [ ]:
import requests
import json

AZURE_ENDPOINT = AZURE_ENDPOINT.rstrip("/")
API_URL = f"{AZURE_ENDPOINT}/providers/microsoft/v1/systemone"

HEADERS = {
    "api-key": AZURE_API_KEY,
    "Content-Type": "application/json",
}

def call_decision_api(state, questions):
    payload = {
        "model": AZURE_DEPLOYMENT_NAME,
        "state": state,
        "questions": questions,
    }

    response = requests.post(
        API_URL,
        headers=HEADERS,
        json=payload,
        timeout=60,
    )

    response.raise_for_status()
    result = response.json()
    print(json.dumps(result, indent=2))
    return result

```raw
1. Noul: binary judgment
Purpose: Estimate the probability that a specific statement is true. Useful for validation gates in your Text-to-SQL pipeline.
Example: Is a generated SQL query aligned with the user's question?
```

In [ ]:
state = {
    "question": "Show total sales for Enfamil in Canada during 2025.",
    "sql": """
        SELECT SUM(pos_gross_sales_amount_usd)
        FROM v_pos_sales_weekly_us_final
        WHERE global_brand = 'Enfamil'
          AND country = 'CA'
          AND week_end_date >= '2025-01-01'
          AND week_end_date <= '2025-12-31';
    """
}

result = call_decision_api(
    state,
    {
        "sql_aligned": {
            "type": "noul",
            "instructions": (
                "Does this SQL query correctly answer the user's question, "
                "including the brand, country, time period, and requested metric?"
            ),
            "criteria": {
                "true": (
                    "All required conditions and the requested metric are correctly represented."
                ),
                "false": "At least one required condition or the metric is incorrect or missing."
            }
        }
    }
)

probability = result["answers"]["sql_aligned"]["noul"]
print("Probability of alignment:", probability)

In [ ]:
state = {
    "user_question": "Show the top retailers.",
    "available_context": "No previous conversation context is available."
}

result = call_decision_api(
    state,
    {
        "next_action": {
            "type": "choice",
            "instructions": (
                "Choose the next action for this Text-to-SQL request. "
                "The KPI and timeframe are unspecified, and no relevant "
                "conversation context is available."
            ),
            "criteria": {
                "generate_sql": (
                    "The request has sufficient information to generate SQL."
                ),
                "ask_clarification": (
                    "Required information is missing and materially affects the result."
                ),
                "reject": (
                    "The request is unsupported or cannot be handled by this system."
                )
            }
        }
    }
)

answer = result["answers"]["next_action"]
print("Selected action:", answer["choice"])
print("Option probabilities:", answer["probabilities"])

In [ ]:
state = {
    "user_question": "Show total sales for Enfamil in Canada during 2025.",
    "sql": """
        SELECT SUM(pos_gross_sales_amount_usd)
        FROM v_pos_sales_weekly_us_final
        WHERE global_brand = 'Enfamil'
          AND country = 'CA'
          AND calendar_year = 2025;
    """
}

result = call_decision_api(
    state,
    {
        "sql_quality": {
            "type": "score",
            "instructions": (
                "Evaluate how well this SQL query answers the user's question. "
                "Consider metric, brand, country, timeframe, and SQL correctness."
            ),
            "criteria": [
                "Poor: Major errors; does not answer the question.",
                "Partial: Some requirements are met, but important errors remain.",
                "Good: Correct intent with minor issues.",
                "Excellent: Correct, complete, and aligned with the question."
            ]
        }
    }
)

answer = result["answers"]["sql_quality"]
print("Score:", answer["score"])
print("Level probabilities:", answer["probabilities"])
print("Rubric:", answer["legend"])

# OLLAMA

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud")

response = llm.invoke("Hi")

print(response.content)

# NEUROMETRIC

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()

base_url = os.getenv("NEUROMETRIC_BASE_URL")
api = os.getenv("NEUROMETRIC_API")

llm = ChatOpenAI(
    model="neurometric/goal-setter",
    api_key=api,          # This sets OPENAI_API_KEY internally
    base_url=base_url
)

messages = [HumanMessage(content="hi?")]
resp = llm.invoke(messages)
print(resp.content)